# Lab 7: Ekspertni sistemi (PyKE-style Forward Chaining)

Budući da `pyke3` nije dostupan putem pip-a za Python 3.12, implementiramo **forward-chaining engine** koji prati isti konceptualni model kao PyKE:
- **Baza činjenica** (`.kfb`) – pohranjuje se kao Python dict/set
- **Baza pravila** (`.krb`) – pravila lančanja unaprijed (foreach/assert logika)
- **Driver** – pokreće engine i postavlja upite


## Forward Chaining Engine (PyKE ekvivalent)

In [ ]:
class KnowledgeEngine:
    """
    Minimalni forward-chaining engine koji oponaša PyKE knowledge_engine.
    Činjenice se čuvaju kao frozenset torki oblika (predikat, arg1, arg2, ...).
    """

    def __init__(self):
        self.facts = set()

    def reset(self):
        self.facts = set()

    def assert_fact(self, predicate, *args):
        """Dodaje novu činjenicu u bazu (THEN / assert)."""
        self.facts.add((predicate,) + tuple(args))

    def query(self, predicate, *pattern):
        """
        Vraća listu tuple-ova koji odgovaraju uzorku.
        None na poziciji argumenta = wildcard (slobodna varijabla).
        """
        results = []
        for fact in self.facts:
            if fact[0] != predicate:
                continue
            args = fact[1:]
            if len(args) != len(pattern):
                continue
            if all(p is None or p == a for p, a in zip(pattern, args)):
                results.append(args)
        return results

    def activate(self, rules):
        """
        Pokreće forward chaining: primjenjuje pravila dok nema novih činjenica.
        rules – lista callable-ova koji primaju engine i vraćaju listu novih činjenica.
        """
        changed = True
        while changed:
            changed = False
            for rule in rules:
                new_facts = rule(self)
                for fact in new_facts:
                    if fact not in self.facts:
                        self.facts.add(fact)
                        changed = True

engine = KnowledgeEngine()
print("KnowledgeEngine spreman.")

## Baza činjenica – `family.kfb`

Svaka činjenica je oblika `son_of(dijete, otac, majka)` ili `daughter_of(dijete, otac, majka)`.

In [ ]:
FAMILY_FACTS = [
    # (predikat, dijete, otac, majka)
    ('son_of',      'bruce',   'thomas',  'norma'),
    ('son_of',      'fred',    'thomas',  'norma'),
    ('daughter_of', 'anne',    'thomas',  'norma'),
    ('son_of',      'homer',   'abner',   'anna'),
    ('daughter_of', 'norma',   'abner',   'anna'),
    ('daughter_of', 'shirley', 'abner',   'anna'),
    ('son_of',      'thomas',  'peter',   'ann'),
    ('daughter_of', 'lucy',    'peter',   'ann'),
    ('son_of',      'brian',   'walter',  'elise'),
    ('daughter_of', 'holly',   'brian',   'shirley'),
    ('son_of',      'mark',    'brian',   'shirley'),
    ('daughter_of', 'jan',     'brian',   'shirley'),
]

def load_family_facts(eng):
    eng.reset()
    for fact in FAMILY_FACTS:
        eng.assert_fact(*fact)
    print(f"Učitano {len(FAMILY_FACTS)} primarnih činjenica.")

load_family_facts(engine)

## Baza pravila – `fc_example.krb` (forward chaining)

Svako pravilo prati PyKE strukturu:
```
foreach  <uvjet>  assert  <nova_cinjenica>
```

In [ ]:
# ── Pravilo 1: son_of  → child_parent (dijete, roditelj, tip_roditelja, tip_djeteta)
def rule_son_of(eng):
    new = []
    for (child, father, mother) in eng.query('son_of', None, None, None):
        new.append(('child_parent', child, father, 'father', 'son'))
        new.append(('child_parent', child, mother, 'mother', 'son'))
    return new

# ── Pravilo 2: daughter_of → child_parent
def rule_daughter_of(eng):
    new = []
    for (child, father, mother) in eng.query('daughter_of', None, None, None):
        new.append(('child_parent', child, father, 'father', 'daughter'))
        new.append(('child_parent', child, mother, 'mother', 'daughter'))
    return new

# ── Pravilo 3: sibling – isti roditelj (otac)
def rule_sibling(eng):
    new = []
    for (c1, parent, _, _) in eng.query('child_parent', None, None, None, None):
        for (c2, p2, _, _) in eng.query('child_parent', None, parent, None, None):
            if c1 != c2:
                new.append(('sibling', c1, c2))
    return new

# ── Pravilo 4: brother – sibling koji je sin
def rule_brother(eng):
    new = []
    for (p1, p2) in eng.query('sibling', None, None):
        # p1 je brat p2 ako je p1 sin nekog roditelja
        if eng.query('child_parent', p1, None, None, 'son'):
            new.append(('brother', p1, p2))
    return new

# ── Pravilo 5: sister – sibling koji je kćerka
def rule_sister(eng):
    new = []
    for (p1, p2) in eng.query('sibling', None, None):
        if eng.query('child_parent', p1, None, None, 'daughter'):
            new.append(('sister', p1, p2))
    return new

# ── Pravilo 6: grandparent – roditelj roditelja
def rule_grandparent(eng):
    new = []
    for (child, parent, _, _) in eng.query('child_parent', None, None, None, None):
        for (_, grandparent, _, _) in eng.query('child_parent', parent, None, None, None):
            new.append(('grandparent', grandparent, child))
    return new

# ── Pravilo 7: how_related (generalna relacija) – koristi izvedene predikate
def rule_how_related(eng):
    new = []
    for (p1, p2) in eng.query('brother', None, None):
        new.append(('how_related', p1, p2, 'brother'))
    for (p1, p2) in eng.query('sister', None, None):
        new.append(('how_related', p1, p2, 'sister'))
    for (gp, gc) in eng.query('grandparent', None, None):
        new.append(('how_related', gp, gc, 'grandparent'))
    # parent
    for (child, parent, ptype, _) in eng.query('child_parent', None, None, None, None):
        label = 'father' if ptype == 'father' else 'mother'
        new.append(('how_related', parent, child, label))
    return new

FC_RULES = [
    rule_son_of, rule_daughter_of, rule_sibling,
    rule_brother, rule_sister, rule_grandparent, rule_how_related
]

# Pokreni forward chaining
load_family_facts(engine)
engine.activate(FC_RULES)
print(f"Ukupno činjenica nakon FC: {len(engine.facts)}")

---
# Zadatak 1 – Ekspertni sistem za porodične relacije

## 1.1 – `fc_test(person)` – kako je osoba povezana sa svima

In [ ]:
def fc_test(person1='bruce'):
    """Ispisuje sve relacije osobe person1 sa ostalim osobama."""
    load_family_facts(engine)
    engine.activate(FC_RULES)

    print(f"\n=== Relacije osobe '{person1}' ===")
    results = engine.query('how_related', person1, None, None)
    if not results:
        print("  Nema pronađenih relacija.")
    for (_, person2, rel) in results:
        print(f"  {person1} i {person2} su: {rel}")

# Test sa zadanom osobom (bruce)
fc_test('bruce')

# Test sa drugom osobom
fc_test('holly')

## 1.2 – Dvije osobe: kako su povezane

In [ ]:
def how_are_related(person1, person2):
    """Ispisuje na koji način su osobe person1 i person2 povezane."""
    load_family_facts(engine)
    engine.activate(FC_RULES)

    print(f"\n=== Relacija između '{person1}' i '{person2}' ===")
    results = engine.query('how_related', person1, person2, None)
    if not results:
        print("  Nema direktne poznate relacije.")
    for (_, _, rel) in results:
        print(f"  {person1} → {person2}: {rel}")

how_are_related('bruce', 'fred')
how_are_related('bruce', 'anne')
how_are_related('thomas', 'holly')

## 1.3 – Sve osobe kojima je proslijeđena osoba **brat**

In [ ]:
def find_siblings_as_brother(person):
    """Vraća sve osobe kojima je 'person' brat."""
    load_family_facts(engine)
    engine.activate(FC_RULES)

    print(f"\n=== '{person}' je brat sljedećim osobama ===")
    results = engine.query('brother', person, None)
    if not results:
        print("  Nema rezultata ili osoba nije brat nijednoj osobi.")
    for (_, sibling) in results:
        print(f"  {person} je brat osobi: {sibling}")

find_siblings_as_brother('bruce')
find_siblings_as_brother('mark')

## 1.4 – Sve osobe kojima je proslijeđena osoba **sestra**

In [ ]:
def find_siblings_as_sister(person):
    """Vraća sve osobe kojima je 'person' sestra."""
    load_family_facts(engine)
    engine.activate(FC_RULES)

    print(f"\n=== '{person}' je sestra sljedećim osobama ===")
    results = engine.query('sister', person, None)
    if not results:
        print("  Nema rezultata ili osoba nije sestra nijednoj osobi.")
    for (_, sibling) in results:
        print(f"  {person} je sestra osobi: {sibling}")

find_siblings_as_sister('anne')
find_siblings_as_sister('holly')

## 1.5 – Sve osobe kojima je proslijeđena osoba **brat ili sestra (sibling)**

In [ ]:
def find_siblings(person):
    """Vraća sve brate i sestre (siblinge) proslijeđene osobe."""
    load_family_facts(engine)
    engine.activate(FC_RULES)

    print(f"\n=== Svi sibling-i osobe '{person}' ===")
    results = engine.query('sibling', person, None)
    if not results:
        print("  Nema sibling-a.")
    for (_, sib) in results:
        print(f"  {person} ↔ {sib}")

find_siblings('bruce')
find_siblings('holly')

## 1.6 – Jesu li dvije osobe **brat/sestra**?

In [ ]:
def are_siblings(person1, person2):
    """Ispisuje poruku samo ako su person1 i person2 sibling-i."""
    load_family_facts(engine)
    engine.activate(FC_RULES)

    result = engine.query('sibling', person1, person2)
    if result:
        print(f"{person1} i {person2} su brat/sestra (sibling).")
    # Ako nisu – ne ispisuje ništa (prema zadatku)

# Jesu sibling
are_siblings('bruce', 'fred')
are_siblings('holly', 'mark')
are_siblings('jan', 'holly')

# Nisu sibling – ne ispisuje se ništa
are_siblings('bruce', 'holly')
are_siblings('thomas', 'norma')
print("(kraj testa – gore nema ispisa za bruce/holly i thomas/norma jer nisu sibling-i)")

## 1.7 – Svi **djedovi i bake (grandparents)** proslijeđene osobe

In [ ]:
def find_grandparents(person):
    """Vraća sve djedove i bake proslijeđene osobe."""
    load_family_facts(engine)
    engine.activate(FC_RULES)

    print(f"\n=== Djedovi/bake osobe '{person}' ===")
    results = engine.query('grandparent', None, person)
    if not results:
        print("  Nema pronađenih djedova/baka.")
    for (gp, _) in results:
        print(f"  {gp} je djed/baka osobi {person}")

find_grandparents('bruce')
find_grandparents('holly')
find_grandparents('mark')

---
# Zadatak 2 – Ekspertni sistem za **vrijeme**

Pravila (forward chaining):
1. `pada_kisa=True` **i** `puse_vjetar=True` → ponesite **kabanicu**
2. `pada_kisa=True` **i** `puse_vjetar=False` → ponesite **kišobran**
3. `pada_kisa=False` **i** `puse_vjetar=False` → ponesite **ništa**

### Baza činjenica (`vrijeme.kfb`)

In [ ]:
# ── Pravila lančanja unaprijed za vremenski ekspertni sistem ──

def weather_rule_kabanica(eng):
    """foreach pada_kisa(True), puse_vjetar(True)  assert  ponijeti(kabanica)"""
    new = []
    if eng.query('pada_kisa', True) and eng.query('puse_vjetar', True):
        new.append(('ponijeti', 'Kabanicu'))
    return new

def weather_rule_kissobran(eng):
    """foreach pada_kisa(True), puse_vjetar(False)  assert  ponijeti(kissobran)"""
    new = []
    if eng.query('pada_kisa', True) and eng.query('puse_vjetar', False):
        new.append(('ponijeti', 'Kišobran'))
    return new

def weather_rule_nista(eng):
    """foreach pada_kisa(False), puse_vjetar(False)  assert  ponijeti(nista)"""
    new = []
    if eng.query('pada_kisa', False) and eng.query('puse_vjetar', False):
        new.append(('ponijeti', 'Ništa'))
    return new

WEATHER_RULES = [weather_rule_kabanica, weather_rule_kissobran, weather_rule_nista]

def weather_driver(pada_kisa: bool, puse_vjetar: bool):
    """Driver za vremenski ekspertni sistem."""
    eng = KnowledgeEngine()
    # Baza činjenica (kfb)
    eng.assert_fact('pada_kisa', pada_kisa)
    eng.assert_fact('puse_vjetar', puse_vjetar)
    # Aktiviraj pravila (fc)
    eng.activate(WEATHER_RULES)
    # Upit: šta ponijeti?
    results = eng.query('ponijeti', None)
    for (item,) in results:
        print(f"Trebate ponijeti: {item}")

print("Vremenski ekspertni sistem spreman.")

### Testiranje vremenskog ekspertnog sistema

In [ ]:
print("--- Scenarij 1: kiša = True, vjetar = True ---")
weather_driver(pada_kisa=True, puse_vjetar=True)

print("\n--- Scenarij 2: kiša = True, vjetar = False ---")
weather_driver(pada_kisa=True, puse_vjetar=False)

print("\n--- Scenarij 3: kiša = False, vjetar = False ---")
weather_driver(pada_kisa=False, puse_vjetar=False)

print("\n--- Scenarij 4: kiša = False, vjetar = True ---")
# Nema eksplicitnog pravila za ovaj slučaj
eng_test = KnowledgeEngine()
eng_test.assert_fact('pada_kisa', False)
eng_test.assert_fact('puse_vjetar', True)
eng_test.activate(WEATHER_RULES)
res = eng_test.query('ponijeti', None)
if res:
    for (item,) in res:
        print(f"Trebate ponijeti: {item}")
else:
    print("Nema preporuke (slučaj nije pokriven pravilima).")

---
## Pregled svih izvedenih činjenica (debug)

Opcijsko – ispisuje sve izvedene relacije za zadanu osobu radi provjere.

In [ ]:
load_family_facts(engine)
engine.activate(FC_RULES)

print("=== Sve 'sibling' relacije ===")
for fact in sorted(f for f in engine.facts if f[0] == 'sibling'):
    print(f"  {fact[1]} ↔ {fact[2]}")

print("\n=== Sve 'grandparent' relacije ===")
for fact in sorted(f for f in engine.facts if f[0] == 'grandparent'):
    print(f"  {fact[1]} je djed/baka osobi {fact[2]}")